# PyTorch to Pega ONNX — pdstools Example

Define a PyTorch model, convert it to Pega-compatible ONNX via `ONNXModel.from_pytorch()`, add metadata, and verify.

## Step 1: Define the PyTorch Model

In [ ]:
import warnings, os, logging

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TORCH_LOGS"] = "-all"
logging.disable(logging.WARNING)

import torch
import torch.nn as nn


class ChurnPredictionModel(nn.Module):
    """Binary classifier for customer churn prediction."""

    def __init__(self, n_features=5):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x)


model = ChurnPredictionModel(n_features=5)
dummy_input = torch.randn(1, 5)
print(f"Model created -- {sum(p.numel() for p in model.parameters()):,} parameters")

Model created -- 737 parameters


## Step 2: Convert to ONNX with `ONNXModel.from_pytorch()`

In [ ]:
import contextlib, io
from pdstools.infinity.resources.prediction_studio.local_model_utils import ONNXModel

# Convert PyTorch model to ONNX with fixed batch size (required by Pega)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    onnx_model = ONNXModel.from_pytorch(
        model,
        dummy_input,
        input_names=["features"],
        output_names=["ChurnProbability"],
        fixed_batch_size=True,
    )

print("ONNX model created with fixed shapes")

5 predictors defined


## Step 3: Attach Pega Metadata and Save

In [ ]:
from pdstools.infinity.resources.prediction_studio.local_model_utils import (
    Metadata, OutcomeType, Output, Predictor,
)

metadata = Metadata(
    type=OutcomeType.BINARY,
    output=Output(label_name="ChurnProbability"),
    predictor_list=[
        Predictor(name="Age",            index=1, input_name="features"),
        Predictor(name="Tenure",         index=2, input_name="features"),
        Predictor(name="MonthlyCharges", index=3, input_name="features"),
        Predictor(name="TotalCharges",   index=4, input_name="features"),
        Predictor(name="NumSupport",     index=5, input_name="features"),
    ],
    py_model_version="1.0.0",
    py_created_by="Falcons",
)

onnx_model.add_metadata(metadata)

output_path = "churn_model_pega.onnx"
onnx_model.save(output_path)
print(f"Saved to {output_path} with {len(metadata.predictor_list)} predictors")

Conversion: passed


## Step 4: Verify the Model

In [ ]:
import onnx, onnxruntime as ort, json, numpy as np

loaded = onnx.load(output_path)

# Shapes must be fixed (no dynamic dims)
print("Input/Output Shapes:")
for inp in loaded.graph.input:
    dims = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"   Input  '{inp.name}': {dims}")
for out in loaded.graph.output:
    dims = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"   Output '{out.name}': {dims}")

# Show embedded pegaMetadata
for prop in loaded.metadata_props:
    if prop.key == "pegaMetadata":
        print("\npegaMetadata:")
        print(json.dumps(json.loads(prop.value), indent=2))

# Run inference
session = ort.InferenceSession(output_path)
test_input = dummy_input.numpy()
outputs = session.run(None, {"features": test_input})
print(f"\nInference: input={test_input[0][:3]}...  output={outputs[0][0]}")
print("\nModel is ready for Pega Prediction Studio.")

Input/Output Shapes:
   Input  'features': [1, 5]
   Output 'ChurnProbability': [1, 1]

pegaMetadata:
{
  "type": "binary",
  "predictorList": [
    {
      "name": "Age",
      "index": 1,
      "inputName": "features",
      "dataType": "Numeric"
    },
    {
      "name": "Tenure",
      "index": 2,
      "inputName": "features",
      "dataType": "Numeric"
    },
    {
      "name": "MonthlyCharges",
      "index": 3,
      "inputName": "features",
      "dataType": "Numeric"
    },
    {
      "name": "TotalCharges",
      "index": 4,
      "inputName": "features",
      "dataType": "Numeric"
    },
    {
      "name": "NumSupport",
      "index": 5,
      "inputName": "features",
      "dataType": "Numeric"
    }
  ],
  "output": {
    "labelName": "ChurnProbability"
  }
}

Test Inference:
   Input:  [ 0.8028426  -0.6825526  -0.20555517]... (first 3 values)
   Output: [0.5373342] (ChurnProbability)

Model is ready for Pega Prediction Studio.
